# Entrenamiento y Evaluación del Modelo (Conv1D-BiLSTM)

Este notebook contiene el pipeline de entrenamiento end-to-end de la red neuronal híbrida **Conv1D + BiLSTM** sobre las ventanas de señal EMG preprocesadas.

---
### Objetivos:
1. Cargar y preparar los datos ventaneados en `DataLoaders` de PyTorch.
2. Entrenar el modelo `Conv1DLSTM` monitoreando las pérdidas (*Loss*) y precisión (*Accuracy*) en entrenamiento y validación.
3. Evaluar el desempeño del modelo sobre el conjunto de test.
4. Generar la **Matriz de Confusión** y el **Reporte de Clasificación** para visualizar la precisión por cada gesto muscular.

In [ ]:
import sys
import os
import yaml
import scipy.io
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar directorio raíz al path
sys.path.append(os.path.abspath(".."))

from src.signal_processing.filters import EMGFilter
from src.signal_processing.feature_extraction import window_signal
from src.models.cnn1d_lstm import Conv1DLSTM
from src.models.evaluate import evaluate_model

# Semilla para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# Selección de dispositivo (CUDA o CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo detectado para entrenamiento: {device}")

In [ ]:
# Cargar configuración global
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_path = "../data/raw/S1_E1_A1.mat"

# Cargar y preprocesar datos
mat = scipy.io.loadmat(data_path)
emg_raw = mat['emg']
restimulus = mat['restimulus'].flatten()
fs = config['dataset']['sample_rate']

filt = EMGFilter(
    sample_rate=fs,
    lowcut=config['processing']['lowcut'],
    highcut=config['processing']['highcut'],
    notch_freq=config['processing']['notch_freq']
)
emg_filtered = filt.process(emg_raw)

# Ventaneo
win_samples = int((config['dataset']['window_size_ms'] / 1000.0) * fs)
step_samples = int(((config['dataset']['window_size_ms'] - config['dataset']['overlap_ms']) / 1000.0) * fs)

X, y = window_signal(emg_filtered, restimulus, win_samples, step_samples)

# Filtrar solo clases objetivo (0 a num_classes - 1)
num_classes = config['dataset']['num_classes']
mask = y < num_classes
X, y = X[mask], y[mask]

# Convertir a Tensores de PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

# División Train (70%), Val (15%), Test (15%)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

# DataLoaders
batch_size = config['model']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Muestras Totales: {len(dataset)}")
print(f"🟢 Train: {len(train_dataset)} | 🟡 Val: {len(val_dataset)} | 🔴 Test: {len(test_dataset)}")

In [ ]:
# Instanciar arquitectura
model = Conv1DLSTM(
    num_channels=config['dataset']['num_channels'],
    num_classes=num_classes,
    conv_channels=config['model']['conv_channels'],
    lstm_hidden_size=config['model']['lstm_hidden_size'],
    lstm_layers=config['model']['lstm_layers'],
    dropout=config['model']['dropout']
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['model']['learning_rate'])

# Contenedores para métricas
epochs = config['model']['epochs']
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print("Iniciando entrenamiento...\n")

for epoch in range(epochs):
    # --- FASE DE ENTRENAMIENTO ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct_train += (preds == targets).sum().item()
        total_train += targets.size(0)
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # --- FASE DE VALIDACIÓN ---
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            val_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_val += (preds == targets).sum().item()
            total_val += targets.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    
    # Guardar métricas
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:.2f}%")

print("\nEntrenamiento completado.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico de Pérdida
ax1.plot(history['train_loss'], label='Train Loss', color='#2b5c8f', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', color='#d95f02', linewidth=2, linestyle='--')
ax1.set_title('Evolución de la Pérdida (Loss)', fontweight='bold')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# Gráfico de Precisión
ax2.plot(history['train_acc'], label='Train Accuracy', color='#2b5c8f', linewidth=2)
ax2.plot(history['val_acc'], label='Val Accuracy', color='#d95f02', linewidth=2, linestyle='--')
ax2.set_title('Evolución de la Precisión (Accuracy)', fontweight='bold')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
class_names = ["Reposo", "Puño", "Pinza", "Mano Abierta", "Apuntar"]

# Generar evaluación sobre conjunto de TEST
cm = evaluate_model(model, test_loader, device, class_names=class_names)

# Graficar Matriz de Confusión
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    xticklabels=class_names, 
    yticklabels=class_names
)
plt.title("Matriz de Confusión - Conjunto de Test", fontsize=13, fontweight='bold')
plt.xlabel("Gesto Predicho por el Modelo")
plt.ylabel("Gesto Real (Ground Truth)")
plt.tight_layout()
plt.show()

In [ ]:
# Guardar el estado del modelo para ser utilizado en main.py y la simulación
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "emg_conv1d_lstm.pth")

torch.save(model.state_dict(), model_path)
print(f"Checkpoint del modelo guardado en: {model_path}")